In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib
from lsff_utils import config_utils
from lsff_utils.results import expand_to_all_scenarios, aggregate_by_scenario

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
scenarios = list(
    config_utils.get_location_fortificant_vehicle_intervention_scenarios()
    .pipe(lambda df: df[(df.location == location) & (df.vehicle == vehicle)])
    .intervention_scenario.unique()
) + ["zero", "baseline"]
scenarios

['intervention', 'zero', 'baseline']

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        ).assign(value=0),
        scenarios,
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,1,intervention,136,0,13.719606
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,2,intervention,136,0,6.260205
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,3,intervention,136,0,13.231798
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,4,intervention,136,0,12.804966
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,5,intervention,136,0,13.069195
...,...,...,...,...,...,...,...,...,...,...,...
1079995,person_time,impairment,anemia,severe,95_plus,severe,1,zero,19,0,0.000000
1079996,person_time,impairment,anemia,severe,95_plus,severe,2,zero,19,0,0.000000
1079997,person_time,impairment,anemia,severe,95_plus,severe,3,zero,19,0,0.000000
1079998,person_time,impairment,anemia,severe,95_plus,severe,4,zero,19,0,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
zero            200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          270000
moderate      270000
not_anemic    270000
severe        270000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.978799e+06
              2                  2.046830e+06
              3                  1.728272e+06
              4                  1.407012e+06
              5                  1.236648e+06
intervention  1                  1.978840e+06
              2                  2.046874e+06
              3                  1.728310e+06
              4                  1.407043e+06
              5                  1.236676e+06
zero          1                  1.978799e+06
              2                  2.046830e+06
              3                  1.728272e+06
              4                  1.407012e+06
              5                  1.236648e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      1                  1.169456e+06
              2                  1.246315e+06
              3                  9.534308e+05
              4                  6.719241e+05
              5                  5.540904e+05
intervention  1                  9.882970e+05
              2                  1.067466e+06
              3                  7.992665e+05
              4                  5.585840e+05
              5                  4.483312e+05
zero          1                  1.169456e+06
              2                  1.246315e+06
              3                  9.534308e+05
              4                  6.719241e+05
              5                  5.540904e+05
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      1                  0.590993
              2                  0.608900
              3                  0.551667
              4                  0.477554
              5                  0.448058
intervention  1                  0.499433
              2                  0.521510
              3                  0.462456
              4                  0.396991
              5                  0.362529
zero          1                  0.590993
              2                  0.608900
              3                  0.551667
              4                  0.477554
              5                  0.448058
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,1,16960.496219
1,Female,0.0,0.019178,not_pregnant,2,16890.884668
2,Female,0.0,0.019178,not_pregnant,3,15946.960379
3,Female,0.0,0.019178,not_pregnant,4,14017.956071
4,Female,0.0,0.019178,not_pregnant,5,13029.810174
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,1,1846.968043
281,Male,95.0,125.000000,not_pregnant,2,1616.329970
282,Male,95.0,125.000000,not_pregnant,3,1661.447043
283,Male,95.0,125.000000,not_pregnant,4,1779.062429


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
1    1.770625e+06
2    1.831699e+06
3    1.543670e+06
4    1.255512e+06
5    1.103242e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.046426e+06
              2                  1.115322e+06
              3                  8.515916e+05
              4                  5.995745e+05
              5                  4.943167e+05
intervention  1                  8.843080e+05
              2                  9.552504e+05
              3                  7.138787e+05
              4                  4.984275e+05
              5                  3.999575e+05
zero          1                  1.046426e+06
              2                  1.115322e+06
              3                  8.515916e+05
              4                  5.995745e+05
              5                  4.943167e+05
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = expand_to_all_scenarios(
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        ).assign(value=0),
        scenarios,
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,1,intervention,136,0,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,2,intervention,136,0,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,3,intervention,136,0,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,4,intervention,136,0,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,5,intervention,136,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
539995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,1,zero,19,0,0.0
539996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,2,zero,19,0,0.0
539997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,3,zero,19,0,0.0
539998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,4,zero,19,0,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      1                  1.959206e+06
              2                  2.033978e+06
              3                  1.664124e+06
              4                  1.291512e+06
              5                  1.133183e+06
intervention  1                  1.877521e+06
              2                  1.950056e+06
              3                  1.593961e+06
              4                  1.239861e+06
              5                  1.086520e+06
zero          1                  1.959206e+06
              2                  2.033978e+06
              3                  1.664124e+06
              4                  1.291512e+06
              5                  1.133183e+06
Name: value, dtype: float64

In [18]:
path = (
    f"./results/{location}/{vehicle}/maternal_disorders_incident_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path).rename(
        columns={"maternal_scenario": "scenario"}
    )
else:
    neonatal_deaths = expand_to_all_scenarios(
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .rename(columns={"maternal_scenario": "scenario"}),
        scenarios,
    )

neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,1,baseline,intervention,73,0,0.000000
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,2,baseline,intervention,73,0,0.000000
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,3,baseline,intervention,73,0,0.000000
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,4,baseline,intervention,73,0,0.000000
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,5,baseline,intervention,73,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
47995,deaths,cause,other_causes,other_causes,18_to_59_months,Male,1,baseline,zero,111,0,147.354336
47996,deaths,cause,other_causes,other_causes,18_to_59_months,Male,2,baseline,zero,111,0,151.594748
47997,deaths,cause,other_causes,other_causes,18_to_59_months,Male,3,baseline,zero,111,0,129.332582
47998,deaths,cause,other_causes,other_causes,18_to_59_months,Male,4,baseline,zero,111,0,99.649695


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      1                  205054.689281
              2                  207305.288236
              3                  172634.615209
              4                  138538.518081
              5                  120916.423670
intervention  1                  204025.329137
              2                  206434.943562
              3                  172095.022714
              4                  138195.044666
              5                  120700.162630
zero          1                  205054.689281
              2                  207305.288236
              3                  172634.615209
              4                  138538.518081
              5                  120916.423670
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = expand_to_all_scenarios(
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/anemia_cases.parquet"
        ).assign(value=0),
        scenarios,
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,1,15276.458404,zero
1,Female,0.0,0.019178,2,14808.048479,zero
2,Female,0.0,0.019178,3,13328.773023,zero
3,Female,0.0,0.019178,4,11611.609394,zero
4,Female,0.0,0.019178,5,9753.105130,zero
...,...,...,...,...,...,...
745,Male,95.0,125.000000,1,1634.000730,intervention
746,Male,95.0,125.000000,2,1404.181104,intervention
747,Male,95.0,125.000000,3,1425.603935,intervention
748,Male,95.0,125.000000,4,1526.505306,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.465356e+07
              2                  2.220715e+07
              3                  2.031122e+07
              4                  1.963280e+07
              5                  1.575827e+07
intervention  1                  2.209124e+07
              2                  1.954621e+07
              3                  1.764825e+07
              4                  1.691805e+07
              5                  1.323257e+07
zero          1                  2.465356e+07
              2                  2.220715e+07
              3                  2.031122e+07
              4                  1.963280e+07
              5                  1.575827e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      1                  2.569999e+07
              2                  2.332247e+07
              3                  2.116281e+07
              4                  2.023238e+07
              5                  1.625258e+07
intervention  1                  2.297555e+07
              2                  2.050146e+07
              3                  1.836212e+07
              4                  1.741647e+07
              5                  1.363252e+07
zero          1                  2.569999e+07
              2                  2.332247e+07
              3                  2.116281e+07
              4                  2.023238e+07
              5                  1.625258e+07
Name: value, dtype: float64

In [25]:
path = f"./results/{location}/{vehicle}/prevalent_anemia_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = expand_to_all_scenarios(
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/ntd_cases_by_scenario.csv"
        ).assign(value=0),
        scenarios,
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
zero          1                  4769.135620
              2                  4876.738006
              3                  4414.789833
              4                  3913.263772
              5                  3437.092295
baseline      1                  4758.801348
              2                  4866.170569
              3                  4405.223395
              4                  3904.784094
              5                  3429.644436
intervention  1                  1674.236104
              2                  1822.857238
              3                  2023.175518
              4                  2025.120335
              5                  1868.076251
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)